# Physics walkthrough

One synthetic muon-like track, a plot at each stage of the pipeline: **deposits → recombination (Q, L) → drift/diffusion/response → wire signals**. Runs on a synthetic event — no external data needed. See the [pipeline overview](../../docs/architecture/pipeline-overview.md) for the diagram this notebook realizes.

In [ ]:
# Resolve the repo root so `import tools` and the relative config/ path work
import os, sys
_d = os.path.abspath(os.getcwd())
while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, 'config')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)

import jax, numpy as np
import matplotlib.pyplot as plt
from tools.simulation import DetectorSimulator
from tools.geometry import generate_detector
from tools.loader import build_deposit_data
from tools.output import to_sparse
from tools.visualization import visualize_wire_signals

detector = generate_detector('config/cubic_wireplane_config.yaml')
sim = DetectorSimulator(detector, include_track_hits=False, include_digitize=True,
                        total_pad=20_000, response_chunk_size=5_000)
cfg = sim.config
sim.warm_up()

## A synthetic track

A single straight MIP-like track (`dE/dx ≈ 2.1 MeV/cm`), positions in mm. `build_deposit_data` splits it into volumes and pads to `total_pad` (see [data model](../../docs/architecture/data-model.md)).

In [ ]:
def straight_track(seed=1, length_cm=150.0, step_cm=0.4):
    rng = np.random.RandomState(seed)
    start = rng.uniform(-150, 150, 3); d = rng.normal(size=3); d /= np.linalg.norm(d)
    n = int(length_cm / step_cm); s = np.arange(n) * step_cm
    pts = np.clip(start[None, :] + s[:, None] * d[None, :], -215.9, 215.9)
    de = np.full(n, 2.1 * step_cm, np.float32)
    dx = np.full(n, step_cm, np.float32)
    tid = np.zeros(n, np.int32)
    return (pts * 10).astype(np.float32), de, dx, tid   # positions in mm

pos_mm, de, dx, tid = straight_track()
deposits = build_deposit_data(pos_mm, de, dx, cfg, track_ids=tid)

## Stage 1 — energy deposits (input)

The raw `(x, y, z, dE, dx)` deposits, before any detector physics.

In [ ]:
fig = plt.figure(figsize=(5, 4)); ax = fig.add_subplot(111, projection='3d')
ax.scatter(pos_mm[:, 0]/10, pos_mm[:, 1]/10, pos_mm[:, 2]/10, c=de, s=4, cmap='viridis')
ax.set_xlabel('x [cm]'); ax.set_ylabel('y [cm]'); ax.set_zlabel('z [cm]')
ax.set_title('Stage 1: energy deposits (dE)'); plt.show()

## Stage 2 — recombination (Q, L)

`process_event_light` runs only the [recombination](../../docs/physics/recombination.md) step: each deposit's energy becomes ionization electrons **Q** and scintillation photons **L** (`Q = N_i·R`, `L = ΔE/W_ph − Q`). For a MIP you expect ~10⁴ e⁻ per 0.4 cm segment.

In [ ]:
light = sim.process_event_light(deposits)
v = next(vi for vi, vol in enumerate(light.volumes) if int(vol.n_actual) > 0)
vol = light.volumes[v]; na = int(vol.n_actual)
Q = np.asarray(vol.charge)[:na]; L = np.asarray(vol.photons)[:na]

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(Q, lw=1); ax[0].set_title('ionization charge Q (e-)'); ax[0].set_xlabel('segment')
ax[1].plot(L, lw=1, color='tab:orange'); ax[1].set_title('scintillation photons L'); ax[1].set_xlabel('segment')
fig.tight_layout(); plt.show()
print(f'Q ~ {Q.mean():.0f} e-/seg,  L ~ {L.mean():.0f} photons/seg')

## Stage 3 — drift, diffusion, response → wire signals

`process_event` runs the full chain: [drift + lifetime attenuation](../../docs/physics/drift-diffusion.md), diffusion-broadened [response kernels](../../docs/physics/response-kernels.md), accumulation, electronics, noise, and digitization. The result is the per-plane wire readout (U/V/Y). Values are in **ADC** after digitization (see [units](../../docs/physics/units.md)).

In [ ]:
signals, _, deposits = sim.process_event(deposits, key=jax.random.PRNGKey(0))
sparse = to_sparse(signals, cfg, threshold_adc=1200 / cfg.electrons_per_adc)
visualize_wire_signals(sparse, cfg, threshold_enc=1200, gamma=0.3, sparse=True)
plt.show()

## Next steps

- [Reading guide](../../docs/architecture/reading-guide.md) — the same flow, function by function
- [Wire vs pixel](../../docs/detector/wire-vs-pixel.md) — the pixel readout path
- `notebooks/getting_started/wire_simulation.ipynb` — truth/track labels